## Image Classification with Pytorch

In [2]:
#from torchvision import datasetsimport 
import torchvision.transforms as transforms
import torch.nn as nn 
import torch
import matplotlib.pyplot as plt
#train_dir = '/data/train'
#train_dataset = ImageFolder(root=train_dir, transform = transforms.ToTensor())

In [3]:
classes = train_dataset.classes
print(classes) # ['cat', 'dog']
print(train_dataset.class_to_idx) # {'cat' : 0, 'dog' : 1}

NameError: name 'train_dataset' is not defined

In [ ]:
# Convolutional layer

class BinaryCNN(nn.Module):
    def __init__(self):
        super(BinaryCNN, self).__init__()

        # Input: 3 RGB channels, output: 16 channels
        self.conv1 = nn.Conv2d(3, 16, kernel_size= 3, stride = 1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten() # tensors flattened into 1 D vector
        self.fc1 = nn.Linear(16 * 112 * 112, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.fc1(self.flatten(x))
        x = self.sigmoid(x)

        return x


#### Extra: For Multi class with CNN

def __init__(self, num_classes):
     super(MultilClassCNN, self).__init__()
     ...
     self.fc = nn.Linear(16*112*112, num_classes)
     self.softmax = nn.Softmax(dim=1)

def forward(self, x):
     ...
     x = self.softmax(x)
     return x

### Convolutional Layers for Images


Grayscale images = 1 channels -> in_channels=1
RGB = in_channels=3
Transparency includes alpha channel = in_channels=4

In [ ]:
# getting the number of channels

from torchvision.transforms import functional 

image = PIL.Image.open("dog.png")
num_channels = functional.get_image_num_channels(image)
print(num_channels)

NameError: name 'PIL' is not defined

Kernel = a convolutional Matrix, dot product of the kernel (green) and the image region - captures image patterns

In Conv layers, the filters are trained based on Data, the number of output channels determined how many filters are applied
- each output channel corresponds to a distinct filter
- higher no. of output channels => layer learns more complex features
- output channel nums are commonly chosen as powers of 2, simplifies the process of combining and dividing channels in subsequent layers




In [ ]:
# Adding Conv layers 

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding = 1)

conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding = 1)

model = Net()
model.add_module('conv2', conv2)

print(model)
# first layer: 3 input channels, 16 output channels
# second layer: 16 output channels, 32 input channels

Net(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


In [ ]:
model.conv2 # can also access each layer indivisually 

Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

In [ ]:
# creating conv blocks - stack layers in a block
# makes our model more flexible to adapt to different datasets
class BinaryImageClassification(nn.Module):
    def __init__(self):
        super(BinaryImageClassification, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        
    # can pass the input to the conv block instead of separate layer
    def forward(self, x): 
        x = self.conv_block(x)
        return x


## Working with pre-trained models
- easier to reused a pre trained model that can be adjusted to the new task (transfer learning)
- difficult to make models from scratch - long process, a lot of ata

In [ ]:
# to load and save a complete pytorch model

torch.save(model.state_dict(), 'BinaryCNN.pth')
               
# Loading Pytorch models - instantiate a new model
new_model = BinaryCNN()

# load saved params
new_model.load_state_dict(torch.load('Binary.CNN.pth'))

NameError: name 'BinaryCNN' is not defined

In [ ]:

# downloading torchvision models (torchvisions provides such)
from torchvision.models import (
    resnet18, ResNet18_Weights # weights = model knowledge
)

# extract weights
weights = ResNet18_Weights.DEFAULT

# instantiate a model passing it weights
model = resnet18(weights=weights)

# store required data transforms
transforms = weights.transforms()

In [ ]:
# Prepare new input images

from PIL import Image

image = Image.open("/Users/srisuphachawla/Downloads/canva-MAHA8XxHPJk.jpg") # load
image_tensor = transforms(image) # tranform
image_reshaped = image_tensor.unsqueeze(0) # reshape

In [ ]:
# Generating a new prediction
model.eval() # evaluation mode for inference

with torch.no_grad(): # disable gradients 
    pred = model(image_reshaped).squeeze(0) # pass image to model and remove batch dimension

pred_cls = pred.softmax(0) # aply softmax
cls_id = pred_cls.argmax().item() # select the highest prob class and extract its index
cls_name = weights.meta["categories"][cls_id] # map class index to label

print(cls_name)  # print class label

tabby


In [ ]:
## example:

# Import resnet18 model
from torchvision.models import (resnet18, ResNet18_Weights)

# Initialize model with default weights
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Set model to evaluation mode
model.eval()

# Initialize the transforms
transform = weights.transforms()

# Apply preprocessing transforms
batch = preprocess(img).unsqueeze(0)

# Apply model with softmax layer
prediction = model(batch).squeeze().softmax(0)

# Apply argmax
class_id = prediction.argmax().item()
score = prediction[class_id].item()
category_name = weights.meta["categories"][class_id]
print(category_name)

NameError: name 'preprocess' is not defined

### Chapter 2: Object Recognition 

##### Bounding Box Representation
- rectangular box described the objects spatial location 

-- to access pics in pytorch, we convert pixels to tensors

----> ToTensor()
- converts pixels to float tensor
Tensor Type:         Torch.float  
Scaled Tensor range: [0.0, 1.0]


----> PILToTensor()
- converts pixels to torch.unit8 (8 bit tensors)
Tensor Type:         Torch.unit8  
Scaled Tensor range: [0, 255]

In [ ]:
import torchvision.transforms as transforms

transform = transform.Compose([
    transforms.Resize(224),
    transforms.ToTensor() # for PIL, only change this to PILToTensor()
    ])

image_tensor = transform(image)

In [ ]:
# drawing the bounding box

from torchvision.utils import draw_bounding_boxes

bbox = torch.tensor([x_min, y_min, x_max, y_max]) # collect coordinates , assume we know them
bbox = bbox.unsqueeze(0) # unsqueeze to two dimensions
bbox_image = draw_bounding_boxes( 
    image_tensor, bbox, width=3, colors="red" # trnasform to image and plot
)

transform = transforms.Compose([
    transforms.ToPILImage() # yes, this is correct
])

pil_image = transform(bbox_image)
plt.imshow(pil_image)


In [ ]:
'''
# Convert bbox into tensors
bbox_tensor = torch.tensor(bbox)

# Add a new batch dimension
bbox_tensor = bbox_tensor.unsqueeze(0)

# Resize image and transform tensor
transform = transforms.Compose([
  transforms.Resize(224),
  transforms.PILToTensor()
])

# Apply transform to image
image_tensor = transform(image)
print(image_tensor)

# Import draw_bounding_boxes
from torchvision.utils import draw_bounding_boxes

# Define the bounding box coordinates
bbox = ([x_min, y_min, x_max, y_max])
bbox_tensor = torch.tensor(bbox).unsqueeze(0)

# Implement draw_bounding_boxes
img_bbox = draw_bounding_boxes(image_tensor, bbox_tensor, width=3, colors="red")

# Tranform tensors to image
transform = transforms.Compose([
    transforms.ToPILImage()
])
plt.imshow(transform(img_bbox))
plt.show()

'''

### Evaluating Object Recognition Model

Intersection over union (IoU)
- IoU = 0, no overlap, 
- IoU = 1, perfect overlap
- commong threshold 0.5 , anything above, is good


In [ ]:
# Intersection over union (IoU)

from torchvision.ops import box_iou

bbox1 = [50, 50 , 150, 150]
bbox2 = [100, 100, 200, 200]

bbox1 = torch.tensor(bbox1).unsqueeze(0)
bbox2 = torch.tensor(bbox2).unsqueeze(0)

iou = box_iou(bbox1, bbox2)
print(iou) # 0.149 - not very accurate prediction


tensor([[0.1429]])


In [ ]:
# Predicting bounding Boxes

model.eval()

with torch.no_grad():
    output = model(input_image)
print(output)
# output - a list of dicts with tensors containing bounding box coordinates,
# shows how confident the model is about each box and predicted class label for each box

boxes = output[0]["boxes"]
scores = output[0]["scores"] # extract confident scores with each score


In [ ]:
# Non- Max Suppression (NMS) -
# Non Max: discards boxes with low confident score to contain an object
# Suppression: discarding boxes with loew IoU


from torchvision.ops import nms

box_indices = nms(
    boxes = boxes, 
    scores = scores,
    iou_threshold=0.5
)
print(box_indices)
filtered = boxes[box_indices] # filter and keep only the selected ones


### Object Detection Using R-CNN
- consists of 3 modules 
1. Generation of Region Proposals - potential bounding boxes
2. Feature Extraction - Convolutional layers
3. Module 3 - Class and Bounding Box Prediction

--> Common to use a pre trained model as the backbone 
Backbone - core CNN architecture responsible for feature extraction

In [ ]:
# Backbone 

'''
from torchvision.models import vgg16, VGG16_Weights

vgg = vgg16(weights=VGG16_Weights.DEFAULT)

backbone = nn.Sequential(
    *list(vgg.features.children())
) 
# nn.Sequential(*list()) - all sublayers are placed into a sequential block as a list, * unpacks the elements  
# .features = only conv layers
# .children() - all layers from block

# R-CNN - Classifier layer

# extract backbone's output size
input_dimension = nn.Sequential(*list(
    vgg_backbone.classifier.children())
)[0].in_features

# create a new classifier
classifier = nn.Sequential(
    nn.Linear(input_dimension, 512),
    nn.ReLU(),
    nn.Linear(512, num_classes)
)

# regressor to predict bounding box coordinates
box_regressor = nn.Sequential(
    nn.Linear(input_dimension, 32),
    nn.ReLU(),
    nn.Linear(32, 4) # 4 outputs for the 4 box coordinates
)

''' 

In [ ]:
# putting the above in one function 

from torchvision.models import vgg16, VGG16_Weights

class ObjectDetectorCNN(nn.module):
    def __init__(self):
        super(ObjectDetectorCNN,self).__init__()
        vgg = vgg16(weights=VGG16_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(vgg.features.children())) 
        input_features = nn.Sequential(*list(vgg.classifier.children()))[0].in_features
        self.classifier = nn.Sequential(
            nn.Linear(input_features, 512),
            nn.ReLU(),
            nn.Linear(512, 2)
        )

        self.box_regressor  = nn.Sequential(
            nn.Linear(input_features, 32),
            nn.ReLU(),
            nn.Linear(32, 4) # 4 outputs for the 4 box coordinates
            )
        
    def forwarrd(self, x):
        features = self.backbone(x)
        bboxes = self.regressor(features)
        classes = self.classfier(features)
        return bboxes, classes

#### Running object recognition 
1. load and transform the image
2. unsqueeze() to add batch dimension 
3. pass the image tensor to the model
4. Run NMS over model's output
5. draw bounding box on top of image

### Region Network Proposals with Faster R-CNN
- smaller area of the image that might contain objects of interest
- backbone, RPN, classifier and Regressor 

Anchor Generator: generate a set of anchor boxes of different sized and aspect ratios

Classifier and Regressor 
- predict if the box contains an object and provide coords

Region of interest (RoI) pooling
- resize the RPN proposal to a fixed size for fully connected layers

In [ ]:
# RPN in pytorch

from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

anchor_generator = AnchorGenerator(
    sizes=((32, 64, 128),),
    aspect_ratios=((0.5, 1.0, 2.0),),
)

roi_pooler = MultiScaleRoIAlign(
    featmap_names=["0"],
    output_size=7, 
    sampling_ratio=2,
)

Fast R-CNN loss functions

RPN classification loss:
- region contains object or not
- binary cross entropy 
- rpn_cls_criterion = nn.BCEWithLogitLoss() (indicates whether a proposed region contains an object)

R-CNN classification loss:
- multiple object classes
- cross-entropy
- rcnn_cls_criterion = nn.CrossEntropyLoss()

RPN box regression loss:
- bounding box coordinates
- mean squared error
- rpn_reg_criterion = nn.MSELoss() (since we may have many classes)

R-CNN box regression loss:
- bounding box coordinates
- mean squared error
- rcnn_reg_criterion = nn.MSELoss()

In [ ]:
# Faster R-CNN 

from torchvision.models.detection import FasterRCNN

backbone = torchvision.models.mobilenet_v2(weights="DEFAULT").features

backbone.out_channels = 1280

model = FasterRCNN(
    backbone=backbone,
    num_classes=num_classes, # define num classes
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler
)


# if load pre trained fater R-CNN
from torchvision.models.detection.faster_rcnn import FasterRCNNPredictor
model = torchvision.models.detection.fastercnn_resnet50_fpn(weights="DEFAULT")

num_classes = 2
in_features = model.roi_heads.box_predictor.cls_score.in_features

# replace model's classifier with a one with the desired num of classes
model.roi_heads.box_predictor = FasterRCNNPredictor(in_features, num_classes)

NameError: name 'torchvision' is not defined